In [24]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import statistics

In [2]:
con = sqlite3.connect("vp_data2_isikud.db")
cur = con.cursor()
cur.execute('ATTACH DATABASE "v33.db" AS v33')

## mis on ptentsiaalsed juured?

ainult obl-id mis on kohakäändes

In [4]:
# failist 001_*

q = """
select * from transactions_verbs_obl_kohakaandes
limit 10
"""
r1 = pd.read_sql_query(q, con)
r1

,head_id,verb,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,kaane
0,2,toimuma,1,lõpp,obl,S,"com,in,sg",UNK,UNK,in
1,3,saama,7,keel,obl,S,"all,com,pl",UNK,UNK,all
2,10,tulema,19,sina,obl,P,"ad,sg",UNK,YES,ad
3,11,viilima,22,tund,obl,S,"com,el,pl",UNK,UNK,el
4,11,viilima,23,juht,obl,S,"ad,com,sg",UNK,YES,ad
5,25,muutuma,40,mis,obl,P,"el,sg",UNK,UNK,el
6,33,minema,59,rahvas,obl,S,"all,com,sg",UNK,UNK,all
7,37,tekkima,69,see,obl,P,"el,sg",UNK,UNK,el
8,51,kutsuma,85,elu,obl,S,"adit,com,sg",UNK,UNK,adit
9,53,tulema,88,toim,obl,S,"adit,com,sg",UNK,UNK,adit


In [33]:
q = """
select distinct root_word, koht, elus, kaane from transactions_verbs_obl_kohakaandes
order by root_word desc
"""
pot_roots1 = pd.read_sql_query(q, con)
pot_roots1

,root_word,koht,elus,kaane
0,ω-linoleenhape,UNK,UNK,ad
1,ω-3-rasvhape,UNK,UNK,el
2,β-glükaan-solubilaa,UNK,UNK,in
3,α-aminohappejääk,UNK,UNK,el
4,ˇkuida,UNK,UNK,in
...,...,...,...,...
401547,%,UNK,UNK,ad
401548,%,UNK,UNK,in
401549,$1,UNK,UNK,ill
401550,$-esi,UNK,UNK,el


In [34]:
len(list(set(list(pot_roots1["root_word"]))))

244257

## distinct juured ja nende esinemiste arv seoses verbidega

In [36]:
# distinct juured ja nende esinemiste arv seoses verbidega
q = """
select distinct root_word, count(*) as cnt from transactions_verbs_obl_kohakaandes
group by root_word
order by cnt desc
"""
pot_roots2 = pd.read_sql_query(q, con)
pot_roots2

,root_word,cnt
0,mina,209164
1,aasta,198619
2,tema,165265
3,sõna,153740
4,see,141815
...,...,...
244252,%-põhimõte,1
244253,%-ilis,1
244254,$1,1
244255,$-esi,1


## mis on mediaan alati sõnade puhul

In [17]:
# alati root+verb+kaane+head_cnt

query = """
SELECT root_word, verb_word||'_'||phrase_case as verb_pat, count(distinct head_id) as head_cnt 
FROM patterns_transaction_isikud_alati
group by root_word, verb_pat
order by head_cnt desc
"""

sb = pd.read_sql_query(query, con)
sb

,root_word,verb_pat,head_cnt
0,mina,meeldima_all,24503
1,mina,olema_ad,12961
2,aasta,saama_ad,10707
3,mina,tulema_ad,9122
4,hääletus,panema_all,8666
...,...,...,...
267816,žüriiliige,minema_all,1
267817,žüriiliige,pakkuma_all,1
267818,žüriiliige,teenima_abl,1
267819,žüriiliige,tulema_all,1


In [25]:
statistics.median(list(sb["head_cnt"]))

1

In [37]:
# alati root+verb+kaane+head_cnt

query = """
SELECT root_word, count(distinct head_id) as head_cnt 
FROM patterns_transaction_isikud_alati
group by root_word
order by head_cnt desc
"""

sb2 = pd.read_sql_query(query, con)
sb2

,root_word,head_cnt
0,mina,127783
1,tema,94822
2,ise,35577
3,sina,32775
4,sõna,27533
...,...,...
72276,0-millennium,1
72277,"0,5",1
72278,-6%,1
72279,-2%,1


In [38]:
statistics.median(list(sb2["head_cnt"]))

1

## kui palju top 1000 root sõna katab 

In [10]:
# alati root + verb_pat count

query = """

select root_word, count(verb_pat) as verb_count from (
SELECT root_word, verb_word||'_'||phrase_case as verb_pat--, count(distinct head_id) as head_cnt 
FROM patterns_transaction_isikud_alati
group by root_word, verb_pat
--order by head_cnt desc
) as tbl1
group by root_word
order by verb_count desc

"""
source = pd.read_sql_query(query, con)
source

,root_word,verb_count
0,tema,402
1,mina,398
2,sina,301
3,kes,282
4,ise,269
...,...,...
72276,0-millennium,1
72277,"0,5",1
72278,-6%,1
72279,-2%,1


In [16]:


a1 = sum(list(source["verb_count"]))
print("kõikide rootide poolt kaetud verbid: ", a1)

a2 =  sum(list(source["verb_count"])[:1000])
print("top 1000 rootide poolt kaetud verbid: ", a2)

print("protsent katvusel?", round(a2*100/a1,2), "%")

kõikide rootide poolt kaetud verbid:  267821
top 1000 rootide poolt kaetud verbid:  64645
protsent katvusel? 24.14 %


## kui palju annoteeritud verb+kääne katab potentsiaalseid juuri?

In [30]:
s3 = pd.read_csv("transactions_verbs_obl_kohakaandes_root_counts_distinct_coverage_v2.csv", sep=";", encoding="utf-8", )
s3

,verb,kaane,root_count,annotated
0,saama,el,21132,True
1,andma,all,12526,True
2,rääkima,el,10930,True
3,jääma,el,9936,True
4,tulema,ad,9166,True
...,...,...,...,...
30081,šveitsima,el,1,False
30082,švipsima,ad,1,False
30083,žestikuleerima,ad,1,False
30084,žisraelima,ad,1,False


In [41]:
s3_2 = s3[s3["annotated"]==True]

In [42]:
kaetud = ["'"+s3_2.iloc[i]["verb"]+"_"+s3_2.iloc[i]["kaane"]+"'" for i in range(len(s3_2))]
kaetudlist = ",".join(kaetud)

In [46]:
q = """
select distinct root_word from transactions_verbs_obl_kohakaandes
where verb||'_'||kaane in ({annlist})
""".format(annlist=kaetudlist)

t1 = pd.read_sql_query(q, con)
t1

,root_word
0,lõpp
1,keel
2,sina
3,mis
4,rahvas
...,...
237076,channeli
237077,pensjoni
237078,kasiioo
237079,lembo


In [ ]:
# verbobl 7.8 milj  kirjet, distinct root 244,257, annoteeritud verb+kääne katab ära 237,081 rooti ???????


In [47]:
con.close()